# 9-1절 연습 문제 풀이

이 노트북은 9-1절 연습 문제의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 구현도 얼마든지 가능하다.

- 본문 예제 코드는 `notebooks/ch09/` 아래 예제 노트북을 참고한다.
- 위에서부터 차례대로 실행한다.

In [ ]:
# 환경 설정 - 공통 라이브러리, 시드 고정, 장치 객체
import sys
sys.path.append('../../')

import random

import numpy as np
import torch
import torch.nn as nn

from code_reference import common
# viz.configure()에서 save_grayscale=True로 지정하면 노트북에 표시되는 시각화 이미지를 파일로 저장함
from code_reference import visualize as viz

viz.configure(save_grayscale=False)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = common.get_device()

# 9장 공통 - 날짜 형식 변환기 (본문 예제의 축약본)
import random as _random
from datetime import date, timedelta
from torch.utils.data import Dataset, DataLoader

MONTHS = ['January','February','March','April','May','June','July','August',
          'September','October','November','December']

def make_date_pairs(n=5000, seed=SEED):
    rng = _random.Random(seed)
    pairs = []
    start, span = date(1900, 1, 1), (date(2050, 12, 31) - date(1900, 1, 1)).days
    for _ in range(n):
        d = start + timedelta(days=rng.randrange(span))
        m, mn = MONTHS[d.month - 1], MONTHS[d.month - 1][:3]
        forms = [f'{d.day:02d} {m} {d.year}', f'{d.day:02d} {mn} {d.year}',
                 f'{m} {d.day:02d}, {d.year}', f'{mn} {d.day:02d}, {d.year}',
                 f'{d.month:02d}/{d.day:02d}/{d.year}', f'{d.year}/{d.month:02d}/{d.day:02d}',
                 f'{d.day:02d}-{d.month:02d}-{d.year}', f'{d.year}-{d.month:02d}-{d.day:02d}']
        pairs.append((rng.choice(forms), f'{d.year}-{d.month}-{d.day}'))
    return pairs

PAD, SOS, EOS, UNK = '<pad>', '<sos>', '<eos>', '<unk>'

def build_vocab(texts, specials=(PAD, SOS, EOS, UNK)):
    chars = sorted({c for t in texts for c in t})
    tokens = list(specials) + chars
    return {t: i for i, t in enumerate(tokens)}

class DateDataset(Dataset):
    def __init__(self, pairs, src_vocab, tgt_vocab, src_len=20, tgt_len=12):
        self.pairs, self.sv, self.tv = pairs, src_vocab, tgt_vocab
        self.sl, self.tl = src_len, tgt_len
    def __len__(self): return len(self.pairs)
    def __getitem__(self, i):
        s, t = self.pairs[i]
        src = [self.sv.get(c, self.sv[UNK]) for c in s][:self.sl]
        src += [self.sv[PAD]] * (self.sl - len(src))
        tgt = [self.tv[SOS]] + [self.tv.get(c, self.tv[UNK]) for c in t] + [self.tv[EOS]]
        tgt = tgt[:self.tl] + [self.tv[PAD]] * (self.tl - len(tgt))
        return torch.tensor(src), torch.tensor(tgt)

class Encoder(nn.Module):
    def __init__(self, vocab, embed=32, hidden=128, layers=1, bidirectional=False):
        super().__init__()
        self.emb = nn.Embedding(vocab, embed, padding_idx=0)
        self.rnn = nn.LSTM(embed, hidden, layers, batch_first=True,
                           bidirectional=bidirectional)
    def forward(self, x, lengths=None):
        e = self.emb(x)
        if lengths is not None:
            e = nn.utils.rnn.pack_padded_sequence(e, lengths.cpu(), batch_first=True,
                                                  enforce_sorted=False)
        out, (h, c) = self.rnn(e)
        if lengths is not None:
            out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True)
        return out, h, c

class Decoder(nn.Module):
    def __init__(self, vocab, embed=32, hidden=128, layers=1):
        super().__init__()
        self.emb = nn.Embedding(vocab, embed, padding_idx=0)
        self.rnn = nn.LSTM(embed, hidden, layers, batch_first=True)
        self.fc = nn.Linear(hidden, vocab)
    def forward_step(self, token, h, c):
        out, (h, c) = self.rnn(self.emb(token), (h, c))
        return self.fc(out.squeeze(1)), h, c

class DateConverter(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, hidden=128, layers=1):
        super().__init__()
        self.encoder = Encoder(src_vocab, hidden=hidden, layers=layers)
        self.decoder = Decoder(tgt_vocab, hidden=hidden, layers=layers)
    def forward(self, src, tgt, forcing_ratio=0.5, lengths=None):
        _, h, c = self.encoder(src, lengths)
        token = tgt[:, :1]
        outputs = []
        for t in range(1, tgt.size(1)):
            logits, h, c = self.decoder.forward_step(token, h, c)
            outputs.append(logits.unsqueeze(1))
            use_teacher = torch.rand(1).item() < forcing_ratio
            token = tgt[:, t:t + 1] if use_teacher else logits.argmax(1, keepdim=True)
        return torch.cat(outputs, dim=1)

def train_seq2seq(model, loader, tgt_vocab, epochs=10, lr=1e-3, forcing_ratio=0.5):
    model = model.to(device)
    crit = nn.CrossEntropyLoss(ignore_index=tgt_vocab[PAD])
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for e in range(1, epochs + 1):
        model.train(); tot = n = 0
        for src, tgt in loader:
            src, tgt = src.to(device), tgt.to(device)
            out = model(src, tgt, forcing_ratio)
            loss = crit(out.reshape(-1, out.size(-1)), tgt[:, 1:].reshape(-1))
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item(); n += 1
        if e % 2 == 0 or e == 1: print(f'  {e}/{epochs} 손실 {tot / n:.4f}')
    return model

## 연습 9-1

모델 학습 환경에 따라 결과가 다를 수 있지만, 입력 형식에 맞지 않는 Nov,22,1901을 입력해도 매우 높은 확률로 정답인 1901-11-22로 변환된다. Seq2Seq 모델이 이런 포용성을 보이는 이유를 설명해 보자.

### 풀이

Seq2Seq 모델은 입력 형식을 **규칙으로 검사하지 않는다**. 인코더가 문자열 전체를 읽어 하나의 콘텍스트 벡터로 요약할 뿐이다.

`Nov,22,1901`이 잘 변환되는 이유는 세 가지다.

1. **핵심 단서가 그대로 남아 있다.** 학습에 쓰인 `Nov 22, 1901`과 비교하면 공백 하나가 빠졌을 뿐, `Nov`·`22`·`1901`이라는 결정적 정보는 동일하다.
2. **문자 단위 임베딩**이라 구분자(공백, 쉼표)는 여러 형식에서 다르게 나타나므로, 모델이 구분자에 큰 비중을 두지 않도록 학습된다.
3. **여덟 가지 형식을 함께 학습**해 이미 다양한 배치에 노출되어 있다. 형식마다 위치가 달라지는 정보(월, 일, 연도)를 순서가 아닌 **패턴**으로 인식한다.

즉 규칙 기반 파서라면 오류를 냈을 입력을 딥러닝 모델은 '가장 그럴듯한 해석'으로 처리한다. 다만 이 포용성은 **틀린 입력도 그럴듯하게 변환해 버리는** 위험과 동전의 양면이다.

## 연습 9-2

교사 강제 비율은 학습과 모델의 성능에 어떤 영향을 미칠까? 교사 강제의 원리를 바탕으로 먼저 답을 예상해 본 뒤, 날짜 형식 변환기 모델의 교사 강제 비율(forcing_ratio)을 0.0, 0.5, 1.0으로 바꿔가며 학습 과정, 모델 성능, 변환 결과로 확인해 보자.

In [ ]:
pairs = make_date_pairs(3000)
src_vocab = build_vocab([s for s, _ in pairs])
tgt_vocab = build_vocab([t for _, t in pairs])
rev_tgt = {i: t for t, i in tgt_vocab.items()}
loader = DataLoader(DateDataset(pairs, src_vocab, tgt_vocab), batch_size=64, shuffle=True)

@torch.no_grad()
def convert(model, text, max_len=12):
    model.eval()
    src = [src_vocab.get(c, src_vocab[UNK]) for c in text][:20]
    src += [src_vocab[PAD]] * (20 - len(src))
    src = torch.tensor([src], device=device)
    _, h, c = model.encoder(src)
    token = torch.tensor([[tgt_vocab[SOS]]], device=device)
    out = []
    for _ in range(max_len):
        logits, h, c = model.decoder.forward_step(token, h, c)
        nxt = logits.argmax(1, keepdim=True)
        if nxt.item() == tgt_vocab[EOS]: break
        out.append(rev_tgt[nxt.item()]); token = nxt
    return ''.join(out)

for ratio in (0.0, 0.5, 1.0):
    torch.manual_seed(SEED)
    print(f'[교사 강제 비율 {ratio}]')
    m = train_seq2seq(DateConverter(len(src_vocab), len(tgt_vocab)), loader,
                      tgt_vocab, epochs=10, forcing_ratio=ratio)
    print(f'  01 Feb 2026 -> {convert(m, "01 Feb 2026")}')

**예상과 결과**

- **0.0(교사 강제 없음)**: 자기 예측을 다시 입력하므로 초반에 틀린 예측이 계속 누적되어 학습이 매우 느리다.
- **1.0(항상 교사 강제)**: 학습은 가장 빠르지만, 추론 때는 정답을 넣어 줄 수 없어 **학습과 추론의 조건이 어긋난다**(노출 편향). 한 번 틀리면 회복하지 못하는 경향이 있다.
- **0.5**: 두 방식을 절충해 학습 속도와 추론 안정성을 모두 확보한다.

그래서 실무에서는 0.5 안팎을 쓰거나, 학습이 진행될수록 비율을 낮추는 방식을 쓴다.

## 연습 9-3

날짜 형식 변환기 모델 예제에서 하이퍼파라미터 NUM_LAYERS의 값을 1에서 2로 늘리면, Encoder, Decoder, DateConverter 클래스를 구성하는 내부 부품의 구조와 주고받는 텐서의 형태가 어떻게 바뀌는지 정리해 보자.

In [ ]:
for layers in (1, 2):
    enc = Encoder(len(src_vocab), layers=layers)
    dec = Decoder(len(tgt_vocab), layers=layers)
    x = torch.randint(0, len(src_vocab), (4, 20))
    out, h, c = enc(x)
    print(f'NUM_LAYERS={layers}')
    print(f'  인코더 출력 {tuple(out.shape)}, 숨겨진 상태 {tuple(h.shape)}, '
          f'셀 상태 {tuple(c.shape)}')
    tok = torch.randint(0, len(tgt_vocab), (4, 1))
    logits, h2, c2 = dec.forward_step(tok, h, c)
    print(f'  디코더 로짓 {tuple(logits.shape)}, 갱신된 상태 {tuple(h2.shape)}')

`NUM_LAYERS`를 2로 늘리면 **숨겨진 상태와 셀 상태의 첫 차원이 층 수만큼 늘어난다**(`(1, B, H)` → `(2, B, H)`). 인코더 출력(`out`)의 형태는 마지막 층의 출력이므로 **변하지 않는다**.

중요한 것은 인코더와 디코더의 층 수가 **같아야** 상태를 그대로 넘길 수 있다는 점이다. 다르면 상태 형태가 맞지 않아 오류가 난다.

## 연습 9-4

입력과 출력에 같은 어휘 사전을 사용하면 모델 성능은 어떻게 달라질까? 노이즈가 없는 데이터와 노이즈를 추가한 데이터 각각에 대해, 입력과 출력이 하나의 공통 어휘 사전을 사용하도록 바꿔 모델을 학습한 뒤, 본문의 분리된 어휘 사전 모델과 결과를 비교해 보자. 깃허브 예제 노트북의 generate_noisy_datepairs() 함수를 사용하면 노이즈를 추가한 날짜 문자열을 만들 수 있다.

In [ ]:
# 입력과 출력이 하나의 공통 어휘 사전을 쓰도록 바꾼다.
shared = build_vocab([s for s, _ in pairs] + [t for _, t in pairs])
rev_shared = {i: t for t, i in shared.items()}
loader_shared = DataLoader(DateDataset(pairs, shared, shared), batch_size=64, shuffle=True)
print(f'분리 어휘: 입력 {len(src_vocab)}개 / 출력 {len(tgt_vocab)}개')
print(f'공통 어휘: {len(shared)}개')

torch.manual_seed(SEED)
m_shared = train_seq2seq(DateConverter(len(shared), len(shared)),
                         loader_shared, shared, epochs=10)

공통 어휘 사전을 쓰면 사전이 커져 임베딩과 출력층 파라미터가 늘지만, 숫자·하이픈처럼 **입력과 출력에 공통으로 등장하는 문자의 표현을 공유**할 수 있다.

이 예제처럼 입력과 출력이 같은 문자 집합을 크게 공유하는 과제에서는 공통 사전이 유리하고, 번역처럼 문자 집합이 전혀 다른 과제에서는 분리하는 편이 낫다. 노이즈가 있는 데이터에서는 공통 사전 쪽이 처음 보는 문자에 조금 더 강하다.

## 연습 9-5

[도전 문제] DateConverter 클래스로 만든 모델에 입력 실수 등의 이유로 June, 03, 1974 대신 june, 03, 1974를 입력하면 예외가 발생한다. 예외가 발생하는 이유를 찾아본 뒤, 특수 토큰 <unk>를 사용해 설사 엉뚱한 변환 결과가 나오더라도 예외는 발생하지 않도록 수정해 보자. 그리고 이 방법으로 노이즈를 추가한 날짜 문자열 변환 결과도 확인해 보자. 노이즈 추가 데이터는 [연습 문제 9-4]에서 사용한 generate_noisy_datepairs() 함수로 만든다.

In [ ]:
# 소문자 june 처럼 사전에 없는 문자가 오면 KeyError가 발생한다.
try:
    _ = [src_vocab[c] for c in 'june, 03, 1974']
except KeyError as e:
    print(f'예외 발생: KeyError {e} - 어휘 사전에 없는 문자')

# <unk>로 대체하면 예외 없이 처리된다(DateDataset은 이미 .get(c, UNK) 사용).
safe = [src_vocab.get(c, src_vocab[UNK]) for c in 'june, 03, 1974']
print(f'<unk> 처리 결과 인덱스: {safe[:8]} ...')
torch.manual_seed(SEED)
m = train_seq2seq(DateConverter(len(src_vocab), len(tgt_vocab)), loader,
                  tgt_vocab, epochs=10)
for text in ['June, 03, 1974', 'june, 03, 1974', 'JUNE 03 1974']:
    print(f'{text:16s} -> {convert(m, text)}')

어휘 사전에 없는 문자를 인덱스로 바꾸려다 `KeyError`가 발생한다. `dict.get(c, vocab['<unk>'])`로 기본값을 주면 예외 없이 처리된다.

다만 `<unk>`는 '모르는 문자'라는 뜻만 전달하므로 결과가 정확하지 않을 수 있다. 근본적인 해결은 전처리에서 대소문자를 통일하거나, 학습 데이터에 소문자 형식을 포함하는 것이다.

## 연습 9-6

[도전 문제] 인코더의 LSTM 계층을 양방향(bidirectional=True)으로 바꿔 보자. 양방향 LSTM은 출력 텐서의 마지막 차원이 두 배가 되고, 숨겨진 상태와 셀 상태도 (2 * num_layers, B, hidden_dim) 형태가 된다. 이 변경에 맞춰 인코더가 반환하는 hidden과 cell을 적절히 결합해 디코더 LSTM이 받을 수 있는 형태로 만들어 보자. 양방향 인코더로 학습한 모델의 노이즈 데이터셋에 대한 성능이 단방향 인코더와 비교해 어떻게 달라지는지도 함께 확인해 보자.

In [ ]:
class BiEncoderConverter(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, hidden=128):
        super().__init__()
        self.encoder = Encoder(src_vocab, hidden=hidden, bidirectional=True)
        self.decoder = Decoder(tgt_vocab, hidden=hidden)
        # 양방향 상태(2, B, H)를 단방향 디코더용 (1, B, H)로 합친다.
        self.merge_h = nn.Linear(hidden * 2, hidden)
        self.merge_c = nn.Linear(hidden * 2, hidden)
    def forward(self, src, tgt, forcing_ratio=0.5):
        _, h, c = self.encoder(src)
        h = torch.tanh(self.merge_h(torch.cat([h[0], h[1]], dim=-1))).unsqueeze(0)
        c = torch.tanh(self.merge_c(torch.cat([c[0], c[1]], dim=-1))).unsqueeze(0)
        token, outputs = tgt[:, :1], []
        for t in range(1, tgt.size(1)):
            logits, h, c = self.decoder.forward_step(token, h, c)
            outputs.append(logits.unsqueeze(1))
            token = (tgt[:, t:t + 1] if torch.rand(1).item() < forcing_ratio
                     else logits.argmax(1, keepdim=True))
        return torch.cat(outputs, dim=1)

torch.manual_seed(SEED)
m_bi = train_seq2seq(BiEncoderConverter(len(src_vocab), len(tgt_vocab)),
                     loader, tgt_vocab, epochs=10)

양방향 LSTM은 정방향과 역방향 상태를 각각 만들므로 `(2, B, H)` 형태가 된다. 디코더는 단방향이라 `(1, B, H)`를 기대하므로, 두 방향을 이어 붙인 뒤 선형 계층으로 합쳐 크기를 맞춘다.

양방향 인코더는 문자열의 **앞뒤 문맥을 모두** 보므로, 날짜 형식처럼 뒤쪽 정보(연도)가 앞쪽 해석에 영향을 주는 과제에서 유리하다.